# 2장 1강: t-검정의 이론과 가정 — 실습문제

## 실습 목표

- 평균 차이와 표준오차를 이용해 t통계량을 직접 계산할 수 있다.
- 단일표본 t검정으로 하나의 표본평균과 기준값을 비교할 수 있다.
- 독립성·정규성·등분산성을 확인한 뒤 독립표본 t검정을 수행할 수 있다.
- 독립표본과 대응표본 상황을 구분하고 적절한 함수를 선택할 수 있다.
- p-value를 유의수준과 비교하여 검정 결과를 올바르게 해석할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `KitchenQual` | 주방 품질 |

> 모든 검정은 양측검정이며 유의수준 `α = 0.05`를 사용합니다.  
> Ames 표본 추출에는 지정된 `random_state`를 사용해 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
import pandas as pd
import scipy.stats

df = pd.read_csv("ames_housing.csv")

df.describe()
print(df.shape)

(1460, 10)


---

## 필수 1. t통계량 직접 계산과 단일표본 t검정

### 문제 1-1. 평균 판매가격은 180,000달러와 다른가?

#### 문제 설명

Ames 주택 30개를 표본으로 추출하여 평균 판매가격이 기준값 180,000달러와 다른지 확인합니다. 먼저 t통계량을 직접 계산한 뒤 `ttest_1samp()` 결과와 비교하세요.

#### 요구사항

1. `SalePrice`에서 `n=30`, `random_state=2`로 표본을 추출하여 `sale_sample`에 저장하세요.
2. 표본 수, 표본평균, 표본표준편차를 출력하세요.
3. `stats.sem()`을 이용해 표준오차를 계산하세요.
4. 다음 식으로 t통계량을 직접 계산하세요.  
   `t = (표본평균 - 기준값) / 표준오차`
5. Shapiro-Wilk 검정으로 표본의 정규성을 확인하세요.
6. 다음 가설을 작성하세요.
   - H₀: 모집단 평균 판매가격은 180,000달러이다.
   - H₁: 모집단 평균 판매가격은 180,000달러가 아니다.
7. `stats.ttest_1samp()`로 단일표본 t검정을 수행하세요.
8. 직접 계산한 t통계량과 함수가 반환한 t통계량을 비교하세요.
9. p-value를 이용해 귀무가설 기각 여부를 판단하세요.

#### 해석 질문

**Q1.** t통계량은 평균 차이와 표준오차를 어떻게 이용한 값인가요?  
**Q2.** 같은 평균 차이라면 표준오차가 작아질수록 t통계량의 절댓값은 어떻게 변하나요?  
**Q3.** 직접 계산한 t통계량과 `ttest_1samp()`의 t통계량은 일치하나요?  
**Q4.** 검정 결과 평균 판매가격이 180,000달러와 다르다고 판단할 수 있나요?

#### 제출 결과

- 기술통계량과 정규성 결과
- 직접 계산한 표준오차와 t통계량
- 단일표본 t검정 결과
- 귀무가설 판단과 해석
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.
import pandas as pd
from scipy import stats

ALPHA = 0.05
MU0 = 180000  # 기준값

df = pd.read_csv("ames_housing.csv")

# 1. 표본 추출
sale_sample = df["SalePrice"].sample(n=30, random_state=2)

# 2. 표본 수, 표본평균, 표본표준편차
n = sale_sample.size
mean = sale_sample.mean()
sd = sale_sample.std(ddof=1)   # 표본표준편차
print(f"표본 수    : {n}")
print(f"표본평균   : {mean:,.2f}")
print(f"표본표준편차: {sd:,.2f}")

# 3. 표준오차
se = stats.sem(sale_sample)
print(f"표준오차   : {se:,.2f}")

# 4. t통계량 직접 계산
t_manual = (mean - MU0) / se
print(f"\n직접 계산한 t = {t_manual:.4f}")

# 5. Shapiro-Wilk 정규성 검정
sw = stats.shapiro(sale_sample)
print(f"\n[Shapiro] W={sw.statistic:.4f}, p={sw.pvalue:.4f}"
      f" → {'정규성 만족' if sw.pvalue > ALPHA else '정규성 위배'}")

# 6. 가설
print("\n[가설]")
print(f"H0 : 모집단 평균 판매가격은 {MU0:,}달러이다.")
print(f"H1 : 모집단 평균 판매가격은 {MU0:,}달러가 아니다. (양측검정)")

# 7. 단일표본 t검정
res = stats.ttest_1samp(sale_sample, popmean=MU0)
print(f"\n[ttest_1samp] t={res.statistic:.4f}, p={res.pvalue:.4f}, df={n-1}")

# 8. 직접 계산값과 비교
print(f"\n차이 = {abs(t_manual - res.statistic):.10f}"
      f" → {'두 값 일치' if abs(t_manual - res.statistic) < 1e-8 else '두 값 불일치'}")

# 9. 기각 여부 판단
if res.pvalue < ALPHA:
    direction = "크다" if mean > MU0 else "작다"
    print(f"\n→ p < {ALPHA} : 귀무가설 기각. 모평균이 {MU0:,}달러와 다르며, "
          f"표본평균으로 보아 더 {direction}고 볼 수 있다.")
else:
    print(f"\n→ p ≥ {ALPHA} : 귀무가설을 기각하지 못한다. "
          f"모평균이 {MU0:,}달러와 다르다고 볼 근거가 부족하다.")

표본 수    : 30
표본평균   : 233,323.80
표본표준편차: 86,013.01
표준오차   : 15,703.76

직접 계산한 t = 3.3956

[Shapiro] W=0.9538, p=0.2133 → 정규성 만족

[가설]
H0 : 모집단 평균 판매가격은 180,000달러이다.
H1 : 모집단 평균 판매가격은 180,000달러가 아니다. (양측검정)

[ttest_1samp] t=3.3956, p=0.0020, df=29

차이 = 0.0000000000 → 두 값 일치

→ p < 0.05 : 귀무가설 기각. 모평균이 180,000달러와 다르며, 표본평균으로 보아 더 크다고 볼 수 있다.


### 필수 1 답변 작성란

- **Q1.**
-> 관측된 평균과 기준값의 차이를 (그 차이의 불확실성을 의미하는) 표준 오차로 나눈 값
- **Q2.**
-> 분모(표준오차)가 작아지므로 T통계량의 절대값은 커진다.
- **Q3.**
-> 일치한다. 두 방법 모두 약 3.3956으로 계산됨

- **Q4.**
-> 그렇다. p-value가 약 0.002로 0.05보다 작아 귀무가설을 기각한다.
-> 따라서 평균 판매가격은 180,000 달러와 다르다

---

## 필수 2. 독립표본 t검정의 가정 점검과 수행

### 문제 2-1. 주방 품질 `Gd`와 `TA` 집단의 평균 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`인 주택과 `TA`인 주택에서 각각 20개를 추출해 평균 판매가격을 비교합니다. 두 집단은 서로 다른 주택으로 구성되어 있습니다.

#### 요구사항

1. `KitchenQual == "Gd"`와 `KitchenQual == "TA"` 집단의 `SalePrice`에서 각각 `n=20`, `random_state=42`로 표본을 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 데이터 수집 구조를 근거로 두 집단의 독립성을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 가정 점검 결과에 따라 `equal_var=True` 또는 `equal_var=False`를 결정하세요.
7. `stats.ttest_ind()`로 독립표본 t검정을 수행하세요.
8. t통계량과 p-value를 출력하고 두 집단 평균 차이를 해석하세요.

#### 해석 질문

**Q1.** 두 집단이 독립표본인 이유는 무엇인가요?
-> GD/TA가 각 집단이 서로 다른 주택으로 구성되어 하나의 관측값이 다른 관측값과 짝을 이루지 않기 때문이다.

**Q2.** 독립성은 별도의 p-value로 확인할 수 있나요?  
-> 일반적으로 독립성은 별ㄷ도의 검정 p-value로 확인하는 것이 아니라
-> 데이터 수집 구조, Feature 엔지니어링 구조를 통해 판단한다.

**Q3.** 정규성과 등분산성 가정은 각각 충족되나요?
-> 필수 2 문제에서는 정규성과 등분산성 위반 근거가 없다.

**Q4.** `equal_var`에는 어떤 값을 사용해야 하나요?  
-> levene p-value가 0.05보다 크므로 True를 사용한다.

**Q5.** 두 집단의 평균 판매가격에는 통계적으로 유의한 차이가 있나요?
-> 유의한 차이가 있다, t 검정 결과 p-value가 약 0.0015fh 0.05보다 작다.
-> 일반적으로 주방설계를 잘해놓으면 주택가격이 달라질 가능성이 있다

#### 제출 결과

- 집단별 표본 수와 평균
- 독립성 설명
- 정규성 및 등분산성 검정 결과
- `equal_var` 선택 근거
- 독립표본 t검정 결과와 해석
- Q1~Q5 답변

In [5]:
# 필수 2 코드를 작성하세요.
import pandas as pd
from scipy import stats

ALPHA = 0.05
df = pd.read_csv("ames_housing.csv")

# 1. 두 집단에서 표본 추출
gd = df.loc[df["KitchenQual"] == "Gd", "SalePrice"].sample(n=20, random_state=42)
ta = df.loc[df["KitchenQual"] == "TA", "SalePrice"].sample(n=20, random_state=42)

# 2. 표본 수와 평균
print(f"Gd : n={gd.size}, 평균={gd.mean():,.1f}, 표준편차={gd.std():,.1f}")
print(f"TA : n={ta.size}, 평균={ta.mean():,.1f}, 표준편차={ta.std():,.1f}")
print(f"평균 차이(Gd-TA) = {gd.mean()-ta.mean():,.1f}")

# 3. 독립성 설명
print("\n[독립성]")
print("KitchenQual은 한 주택에 하나의 등급만 부여되므로 Gd 집단과 TA 집단에")
print("같은 주택이 중복 포함될 수 없다. 두 표본은 서로 다른 행에서 추출되었고")
print("한쪽 값이 다른 쪽 값에 영향을 주지 않으므로 독립집단이다.")

# 4. Shapiro-Wilk 정규성 검정
sw_gd = stats.shapiro(gd)
sw_ta = stats.shapiro(ta)
print(f"\n[Shapiro] Gd : W={sw_gd.statistic:.4f}, p={sw_gd.pvalue:.4f}"
      f" → {'정규성 만족' if sw_gd.pvalue > ALPHA else '정규성 위배'}")
print(f"[Shapiro] TA : W={sw_ta.statistic:.4f}, p={sw_ta.pvalue:.4f}"
      f" → {'정규성 만족' if sw_ta.pvalue > ALPHA else '정규성 위배'}")

# 5. Levene 등분산 검정
lev = stats.levene(gd, ta, center="median")
print(f"\n[Levene] 통계량={lev.statistic:.4f}, p={lev.pvalue:.4f}")

# 6. equal_var 결정
equal_var = lev.pvalue > ALPHA
method = "독립표본 t검정 (Student)" if equal_var else "Welch t검정"
print(f"→ equal_var = {equal_var} : {method} 사용")

# 7. 독립표본 t검정
res = stats.ttest_ind(gd, ta, equal_var=equal_var)

# 8. 결과 출력 및 해석
print(f"\n[{method}]")
print(f"t = {res.statistic:.4f}")
print(f"p = {res.pvalue:.4f}")

if res.pvalue < ALPHA:
    higher = "Gd" if gd.mean() > ta.mean() else "TA"
    print(f"\n→ p < {ALPHA} : 귀무가설(두 집단 평균이 같다)을 기각한다.")
    print(f"   주방 등급에 따른 판매가격 차이는 통계적으로 유의하며, "
          f"{higher} 집단의 평균이 더 높다.")
else:
    print(f"\n→ p ≥ {ALPHA} : 귀무가설을 기각하지 못한다.")
    print("   두 집단의 평균 판매가격 차이가 유의하다고 보기 어렵다.")

# 정규성이 깨졌을 때를 위한 보조 검정
if sw_gd.pvalue <= ALPHA or sw_ta.pvalue <= ALPHA:
    mw = stats.mannwhitneyu(gd, ta, alternative="two-sided")
    print(f"\n[보조] Mann-Whitney U={mw.statistic:.4f}, p={mw.pvalue:.4f}")

Gd : n=20, 평균=191,916.6, 표준편차=58,735.7
TA : n=20, 평균=137,432.5, 표준편차=40,145.3
평균 차이(Gd-TA) = 54,484.1

[독립성]
KitchenQual은 한 주택에 하나의 등급만 부여되므로 Gd 집단과 TA 집단에
같은 주택이 중복 포함될 수 없다. 두 표본은 서로 다른 행에서 추출되었고
한쪽 값이 다른 쪽 값에 영향을 주지 않으므로 독립집단이다.

[Shapiro] Gd : W=0.9555, p=0.4580 → 정규성 만족
[Shapiro] TA : W=0.9410, p=0.2503 → 정규성 만족

[Levene] 통계량=3.7806, p=0.0593
→ equal_var = True : 독립표본 t검정 (Student) 사용

[독립표본 t검정 (Student)]
t = 3.4249
p = 0.0015

→ p < 0.05 : 귀무가설(두 집단 평균이 같다)을 기각한다.
   주방 등급에 따른 판매가격 차이는 통계적으로 유의하며, Gd 집단의 평균이 더 높다.


### 필수 2 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**
- **Q5.**

---

## 과제. 대응표본과 독립표본 상황 구분

### 문제 3-1. 같은 주택의 보수 전후 예상 판매가격 비교

#### 문제 설명

부동산 회사가 동일한 주택 10채에 대해 보수 전 예상 판매가격과 보수 후 예상 판매가격을 각각 산정했습니다. 같은 위치의 값은 동일한 주택의 전후 가격으로 서로 짝을 이룹니다.

```python
before_price = [145, 162, 178, 155, 190, 172, 168, 181, 159, 175]
after_price  = [154, 170, 185, 164, 201, 179, 176, 190, 168, 183]
```

단위는 천 달러입니다.

#### 요구사항

1. 두 배열의 길이가 같은지 확인하세요.
2. 보수 전후 평균을 계산하세요.
3. 독립표본 t검정과 대응표본 t검정 중 적절한 방법을 선택하고 이유를 설명하세요.
4. 대응표본 t검정의 정규성 가정은 개별 배열이 아니라 `after_price - before_price` 차이값에 적용된다는 점을 확인하세요.
5. 차이값에 Shapiro-Wilk 정규성 검정을 수행하세요.
6. 적절한 t검정을 실행하고 t통계량과 p-value를 출력하세요.
7. 보수 전후 평균 예상 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 이 데이터는 독립표본인가요, 대응표본인가요?  
**Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요?  
**Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?  
**Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?

#### 제출 결과

- 표본 관계 판단과 근거
- 전후 평균과 차이값
- 차이값의 정규성 결과
- 대응표본 t검정 결과
- 결과 해석
- Q1~Q4 답변

In [1]:
# 과제 코드를 작성하세요.

import numpy as np
from scipy import stats

before_price = [145, 162, 178, 155, 190, 172, 168, 181, 159, 175]
after_price  = [154, 170, 185, 164, 201, 179, 176, 190, 168, 183]

before = np.array(before_price, dtype=float)
after  = np.array(after_price, dtype=float)

# 1. 길이 확인
print("1) 길이 확인")
print(f"   보수 전: {len(before)}채, 보수 후: {len(after)}채")
if len(before) != len(after):
    raise ValueError("두 배열의 길이가 다릅니다. 대응표본 분석 불가.")
print("   -> 길이가 같으므로 짝을 지을 수 있습니다.\n")

# 2. 평균
print("2) 평균")
print(f"   보수 전 평균: {before.mean():.2f} 천 달러")
print(f"   보수 후 평균: {after.mean():.2f} 천 달러")
print(f"   평균 차이   : {after.mean() - before.mean():.2f} 천 달러\n")

# 3~4. 차이값 (정규성 가정은 개별 배열이 아니라 이 차이값에 적용)
diff = after - before
print("3~4) 차이값 (보수 후 - 보수 전)")
print(f"   {diff.astype(int).tolist()}")
print(f"   평균 {diff.mean():.2f}, 표준편차 {diff.std(ddof=1):.3f}\n")

# 5. Shapiro-Wilk 정규성 검정 (차이값 대상)
w, p_shapiro = stats.shapiro(diff)
print("5) Shapiro-Wilk 정규성 검정 (차이값)")
print(f"   W = {w:.4f}, p-value = {p_shapiro:.4f}")
print("   -> 정규성 " + ("충족" if p_shapiro > 0.05 else "위배") + " (유의수준 0.05)\n")

# 6. 대응표본 t검정
t_stat, p_value = stats.ttest_rel(after, before)
print("6) 대응표본 t검정")
print(f"   t통계량 = {t_stat:.4f}")
print(f"   p-value = {p_value:.10f}")
print(f"   자유도  = {len(diff) - 1}\n")

# 7. 결론
alpha = 0.05
print("7) 결론")
if p_value < alpha:
    print(f"   p-value가 {alpha}보다 작으므로 보수 전후 평균 예상 판매가격에")
    print(f"   유의한 차이가 있습니다. 보수 후가 평균 {diff.mean():.2f} 천 달러 높습니다.")
else:
    print(f"   p-value가 {alpha}보다 크므로 유의한 차이가 있다고 볼 수 없습니다.")

# 참고: 정규성이 깨졌을 때의 비모수 대안
w_stat, p_wilcoxon = stats.wilcoxon(after, before)
print(f"\n참고) Wilcoxon 부호순위검정: 통계량 {w_stat:.1f}, p-value {p_wilcoxon:.4f}")

1) 길이 확인
   보수 전: 10채, 보수 후: 10채
   -> 길이가 같으므로 짝을 지을 수 있습니다.

2) 평균
   보수 전 평균: 168.50 천 달러
   보수 후 평균: 177.00 천 달러
   평균 차이   : 8.50 천 달러

3~4) 차이값 (보수 후 - 보수 전)
   [9, 8, 7, 9, 11, 7, 8, 9, 9, 8]
   평균 8.50, 표준편차 1.179

5) Shapiro-Wilk 정규성 검정 (차이값)
   W = 0.8871, p-value = 0.1574
   -> 정규성 충족 (유의수준 0.05)

6) 대응표본 t검정
   t통계량 = 22.8079
   p-value = 0.0000000028
   자유도  = 9

7) 결론
   p-value가 0.05보다 작으므로 보수 전후 평균 예상 판매가격에
   유의한 차이가 있습니다. 보수 후가 평균 8.50 천 달러 높습니다.

참고) Wilcoxon 부호순위검정: 통계량 0.0, p-value 0.0020


### 과제 답변 작성란

- **Q1.** 이 데이터는 독립표본인가요, 대응표본인가요?
-> 대응표본이다.

- **Q2.** 대응표본 t검정에서는 무엇의 정규성을 확인해야 하나요?
-> 차이값인 (보수 후 - 보수 전)의 정규성을 확인하면 된다.

- **Q3.** `stats.ttest_ind()`가 아니라 어떤 함수를 사용해야 하나요?
-> stats.ttest_rel(이전 데이터, 이후 데이터) 

- **Q4.** 검정 결과 보수 전후 예상 판매가격에 유의한 차이가 있나요?
-> p-value가 0.05보다 작으므로 보수 전후 평균 예상 판매가격에 유의한 차이가 있다.

---

## 실습 마무리

1. t통계량은 어떤 두 값을 비교하여 계산하나요?
-> 관측된 평균 차이를 표준오차(그 차이의 불확실성)와 비교한다.

2. 단일표본 t검정은 어떤 상황에서 사용하나요?
-> 하나의 표본평균을 특정 기준값 또는 모집단 평균과 비교할때 사용한다.

3. 독립표본과 대응표본은 데이터 수집 구조에서 어떤 차이가 있나요?
-> 독립표본은 서로 다른 대상들로 구성되고, 대응표본은 같은 대상의 반복 측정값처럼 각 관측값이 짝을 이룬다.

4. 독립표본 t검정에서 정규성·등분산성·독립성은 각각 어떻게 확인하나요?
-> Shapiro, Levene, 독립성은 데이터 수집 구조

5. 분산이 같다고 보기 어렵거나 확신할 수 없을 때 어떤 t검정을 사용할 수 있나요?
-> 